# Clase 025 — Datos faltantes

**Parte 0** · VanderPlas cap. 3 § 3.5.

> 🎯 Detectar, cuantificar y manejar NaN con criterio. Eliminar no siempre es la respuesta.

> ⏱️ ~75 min

## ⚙️ Setup

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)

## 1️⃣ Tipos de missing en pandas

| Valor | Tipo | Caso |
|---|---|---|
| `np.nan` (float NaN) | float | numérico clásico — incluso en columnas "int" |
| `None` | object | strings y mixto |
| `pd.NaT` | datetime | timestamps |
| `pd.NA` | NA-aware | dtypes nullable (Int64, boolean, string) |

**Gotcha histórico**: una columna `int` con NaN se promueve a `float64` (porque NumPy int no tiene NaN). Solución: usa `pd.Int64` (nullable).

## 2️⃣ Detección y cuantificación

In [ ]:
df = pd.DataFrame({
    'a': [1, 2, np.nan, 4, 5],
    'b': [np.nan, 'x', 'y', np.nan, 'z'],
    'c': [10, 20, 30, 40, 50],
})

print('isna() (matriz bool):')
print(df.isna())
print()
print('Por columna (count NaN):')
print(df.isna().sum())
print()
print('% NaN por columna:')
print((df.isna().mean() * 100).round(1))
print()
print('Filas con al menos un NaN:')
print(df.isna().any(axis=1).sum())

## 3️⃣ `dropna` — eliminar con precisión

```python
df.dropna()                            # filas con cualquier NaN
df.dropna(how='all')                   # solo filas con TODOS NaN
df.dropna(thresh=2)                    # mantén filas con >= 2 no-NaN
df.dropna(subset=['col_clave'])        # NaN solo en esta col cuenta
df.dropna(axis=1)                      # eliminar COLUMNAS con NaN
```

In [ ]:
print(f'shape original           : {df.shape}')
print(f'dropna() any            : {df.dropna().shape}')
print(f'dropna(subset=["a"])    : {df.dropna(subset=["a"]).shape}')
print(f'dropna(axis=1)          : {df.dropna(axis=1).shape}  ← elimina cols b'); print()
print(df.dropna(subset=['a']))

## 4️⃣ `fillna` — imputar

```python
df.fillna(0)                           # constante
df.fillna({'a': 0, 'b': 'missing'})    # por columna
df.fillna(df.mean(numeric_only=True))  # media por columna
df['col'].fillna(df['col'].median())   # mediana en una col
df['x'].ffill()                        # último válido hacia adelante
df['x'].bfill()                        # próximo válido hacia atrás
df['x'].interpolate(method='linear')   # interpolación
```

In [ ]:
# Imputar con dict por columna
df2 = df.fillna({'a': df['a'].median(), 'b': 'desconocido'})
print(df2)

## 5️⃣ Imputación por grupo — la opción correcta

Imputar con la media **global** introduce sesgo. Imputar con la **media del grupo** (por especie, por región, etc.) es mucho mejor:

In [ ]:
# Demo: imputar nota por curso (grupo)
notas = pd.DataFrame({
    'curso': ['A', 'A', 'A', 'B', 'B', 'B', 'C', 'C'],
    'nota' : [7.0, 8.0, np.nan, 4.0, 5.0, np.nan, 9.0, np.nan],
})

# Mediana por grupo
notas['nota_imp'] = notas.groupby('curso')['nota'].transform(
    lambda s: s.fillna(s.median())
)
print(notas)
print('\nMediana global =', notas['nota'].median(), '← sería sesgada por A y C')

## 6️⃣ `was_missing` como feature

A veces el faltante es **señal** (ej: respuesta opcional → "prefiero no contestar"). Antes de imputar, guarda un flag:

In [ ]:
df3 = df.copy()
df3['a_was_missing'] = df3['a'].isna()
df3['a'] = df3['a'].fillna(df3['a'].mean())
print(df3)

## 7️⃣ Dtypes nullable (modernos)

Desde pandas 1.0+, hay dtypes que soportan NaN sin promover a float:

In [ ]:
# int normal: NaN promueve a float
s_old = pd.Series([1, 2, np.nan, 4])
print(f'dtype int+NaN antiguo : {s_old.dtype}')   # float64

# int nullable: mantiene int
s_new = pd.Series([1, 2, pd.NA, 4], dtype='Int64')
print(f'dtype Int64 (nullable): {s_new.dtype}')   # Int64
print(s_new)

## 🎯 Cómo decidir: eliminar, imputar o flag

| Situación | Estrategia |
|---|---|
| <1% missing, distribución aleatoria | `dropna` (rápido, bajo costo) |
| 5-30% missing, MCAR (Missing Completely At Random) | imputar (media/mediana global) |
| Missing por grupo (especie, región) | imputar por grupo (`groupby.transform`) |
| Missing es informativo (encuesta opcional) | imputar + flag `was_missing` |
| >50% missing | considerar eliminar la columna |
| Series temporal | `ffill`/`interpolate` |

## ✅ Checklist

- [ ] Cuantifico NaN antes de actuar (% por col y fila)
- [ ] Uso `dropna(subset=...)` en vez de `dropna()` ciego
- [ ] Imputo por grupo cuando hay estructura
- [ ] Creo flag `was_missing` cuando el missing es señal
- [ ] Uso `Int64` para columnas enteras con NaN

## 📝 Homework

Ver `README.md`. Reporte missing penguins, 3 estrategias dropna/impute, flag was_missing, Int64.

## 📖 Definiciones y características

**`NaN` (Not a Number)**

Float especial IEEE 754 que representa missing en columnas numéricas. Característica: NO es igual a sí mismo (`np.nan == np.nan` → False). Usa `pd.isna()` para detectarlo.

**`None` vs `NaN`**

**`None`** es objeto Python (en columnas `object`). **`NaN`** es float (en columnas numéricas). Pandas trata ambos como missing pero internamente son distintos.

**`pd.NA`**

Sentinel "missing universal" (pandas 1.0+) compatible con dtypes nullable (`Int64`, `boolean`, `string`). Comportamiento más consistente que NaN en operaciones.

**MCAR / MAR / MNAR**

**MCAR** (Missing Completely At Random): missing es aleatorio. **MAR** (Missing At Random): missing depende de variables observadas. **MNAR** (Missing Not At Random): depende del propio valor missing (peor caso).

**Imputación**

Reemplazar missing con un valor estimado. Estrategias: media/mediana/moda global, por grupo (`groupby.transform`), forward-fill (series temporales), KNN, regresión, MICE.

**`was_missing` flag**

Columna booleana adicional que registra qué filas tenían missing antes de imputar. Permite al modelo usar el hecho de que faltaba como feature (a veces es informativo).

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `if df['col'] == np.nan` no funciona | NaN no es igual a nada (incluso a sí mismo). **Fix**: `df['col'].isna()` o `pd.isna(df['col'])`. |
| `df.dropna()` borra todo | Sin parámetros, borra fila con CUALQUIER NaN. Si tienes muchos NaN repartidos, te queda 0 filas. **Fix**: `dropna(subset=['col_clave'])` para ser selectivo. |
| Imputar con media global introduce sesgo | Si los grupos tienen medias muy distintas, imputar global aplana las diferencias. **Fix**: imputa por grupo con `groupby.transform(lambda s: s.fillna(s.median()))`. |
| Una columna `int` se volvió `float` tras leer CSV | Tiene NaN → promoción automática (NumPy int no soporta NaN). **Fix**: dtype nullable `Int64`: `df['col'] = df['col'].astype('Int64')`. |
| `fillna(0)` mata el flag de "missing" | Pierdes la información de que faltaba. **Fix**: crea `df['col_was_missing'] = df['col'].isna()` ANTES de imputar. |

## ❓ Preguntas frecuentes

**❓ ¿Eliminar, imputar o flag?**

Depende: **<1% missing aleatorio** → drop. **5-30% MAR** → imputar (mediana por grupo). **>50%** → eliminar columna o tratar como categoría 'missing'. **Si missing es informativo** → flag + imputar.

**❓ ¿Mediana o media para imputar?**

**Mediana** es más robusta a outliers (recomendada por default). **Media** si la distribución es ~normal y sin outliers. Para categóricas: **moda**.

**❓ ¿`ffill` siempre bueno para series temporales?**

Bueno cuando los valores cambian poco entre observaciones (precios, temperaturas). Malo si los gaps son largos o el dato es volátil. Considera `interpolate(method='time')` para mejor resultado.

**❓ ¿Cómo sé si la imputación afecta mi modelo?**

Compara métricas: (a) baseline drop, (b) imputación X, (c) imputación + flag. Si el flag mejora el modelo, el missing era informativo (caso MNAR).

**❓ ¿KNN/MICE para imputación en pandas?**

No nativo. Usa `sklearn.impute.KNNImputer` o `IterativeImputer` (= MICE). Más caro pero mejor que mediana en datasets con correlaciones fuertes.

## 🔗 Referencias

- VanderPlas cap. 3 § 3.5
- [Missing data user guide](https://pandas.pydata.org/docs/user_guide/missing_data.html)

➡️ **Siguiente:** [026 — MultiIndex](../026-pandas-multiindex/README.md)